In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
# %pip install xlrd -q



In [ ]:
fred_api_key = notebookutils.credentials.getSecret("https://ustrat-vault.vault.azure.net/", "fred-api-key")


In [ ]:
import requests
from datetime import datetime, timezone
from pyspark.sql import Row

FRED_SERIES = {
    "ISRATIO": "total_business_inventories_sales_ratio",
    "MRTSIM4400AUSS": "retail_inventories_ex_motor_vehicle",
    "NEWORDER": "mfrs_new_orders_nondefense_cap_goods_ex_aircraft",
    "TCU": "capacity_utilization_total",
    "PPIFIS": "ppi_final_demand",
    "PCU484484": "ppi_truck_transportation",
    "PCU4931149311": "ppi_warehousing_storage",
    "CES4300000003": "avg_hourly_earnings_transportation_warehousing",
    "TSIFRGHT": "dot_freight_transportation_services_index",
}

ingested_at = datetime.now(timezone.utc).isoformat()
rows = []

for series_id, metric_name in FRED_SERIES.items():
    resp = requests.get(
        "https://api.stlouisfed.org/fred/series/observations",
        params={"series_id": series_id, "api_key": fred_api_key, "file_type": "json"},
        timeout=30,
    )
    resp.raise_for_status()
    for obs in resp.json().get("observations", []):
        if obs["value"] == ".":
            continue
        rows.append(Row(
            series_id=series_id,
            metric_name=metric_name,
            observation_date=obs["date"],
            value=float(obs["value"]),
            source="FRED",
            ingested_at=ingested_at,
        ))

print(f"Pulled {len(rows)} observations across {len(FRED_SERIES)} series")


In [ ]:
import xlrd

xls_bytes = requests.get(
    "https://www.newyorkfed.org/medialibrary/research/interactives/gscpi/downloads/gscpi_data.xlsx",
    timeout=30,
).content
wb = xlrd.open_workbook(file_contents=xls_bytes)
ws = wb.sheet_by_name("GSCPI Monthly Data")

for r in range(5, ws.nrows):
    date_str = ws.cell_value(r, 0)
    value = ws.cell_value(r, 1)
    if not date_str or value == "":
        continue
    obs_date = datetime.strptime(date_str, "%d-%b-%Y").strftime("%Y-%m-%d")
    rows.append(Row(
        series_id="GSCPI",
        metric_name="global_supply_chain_pressure_index",
        observation_date=obs_date,
        value=float(value),
        source="NY_FED",
        ingested_at=ingested_at,
    ))

print(f"Total rows now: {len(rows)}")


In [ ]:
df = spark.createDataFrame(rows)
df.write.mode("overwrite").format("delta").saveAsTable("supply_chain_indicators")

display(df.orderBy("series_id", "observation_date"))
print(f"Wrote {df.count()} rows across {len(FRED_SERIES) + 1} series to supply_chain_indicators")


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

COSMOS_ENDPOINT = "https://bc836f6d-54c1-43a9-b890-87cdb4428a90.zbc.sql.cosmos.fabric.microsoft.com:443/"
COSMOS_DATABASE = "operational_db"
COSMOS_CONTAINER = "supply_chain_dashboard"

config_write = {
    "spark.cosmos.accountendpoint": COSMOS_ENDPOINT,
    "spark.cosmos.database": COSMOS_DATABASE,
    "spark.cosmos.container": COSMOS_CONTAINER,
    "spark.cosmos.write.strategy": "ItemOverwrite",
    "spark.cosmos.accountDataResolverServiceName": "com.azure.cosmos.spark.fabric.FabricAccountDataResolver",
    "spark.cosmos.auth.type": "AccessToken",
    "spark.cosmos.useGatewayMode": "true",
    "spark.cosmos.auth.aad.audience": "https://cosmos.azure.com/",
}

# Get the latest observation per series_id from the Lakehouse table
indicators_df = spark.table("supply_chain_indicators")
window = Window.partitionBy("series_id").orderBy(F.col("observation_date").desc())

latest_df = (
    indicators_df
    .withColumn("rn", F.row_number().over(window))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .withColumn("id", F.col("series_id"))  # Cosmos DB requires an 'id' field
)

latest_df.write.format("cosmos.oltp").options(**config_write).mode("APPEND").save()

print(f"Synced {latest_df.count()} latest indicator values to Cosmos DB")

# Restricted container: same snapshot, filtered to the rows the NYFedOnlyReader
# Lakehouse role allows. Keep this predicate in sync with that role's filter.
RESTRICTED_SOURCE = "NY_FED"

config_write_restricted = {**config_write, "spark.cosmos.container": "supply_chain_dashboard_restricted"}

restricted_df = latest_df.filter(F.col("source") == RESTRICTED_SOURCE)
restricted_df.write.format("cosmos.oltp").options(**config_write_restricted).mode("APPEND").save()

print(f"Synced {restricted_df.count()} rows to supply_chain_dashboard_restricted")

